# Ingesting GDPR Chapter 3 and Retail Privacy Policy from Volume

In [0]:
import pandas as pd
import re
import os

# Define the updated corporate domain namespace
SCHEMA_PATH = "main.default"
VOLUME_PATH = f"/Volumes/main/default/compliance_sources"

legislation_file = f"{VOLUME_PATH}/GDPR_Chapter_3.md"
policy_file = f"{VOLUME_PATH}/retail_privacy_policy.md"

# Verify Volume access before executing the pipeline
for path in [legislation_file, policy_file]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing asset in Unity Catalog Volume: {path}")

print(f"Verified secure namespace access at {VOLUME_PATH}. Processing data assets...")

# ───────────────────────────────────────────────────────────────────────────────────────
# 1. PROCESS STATUTORY LEGISLATION LAYER
# ───────────────────────────────────────────────────────────────────────────────────────
with open(legislation_file, "r", encoding="utf-8") as f:
    raw_leg_text = f.read()

leg_segments = re.split(r'(?=\n##\s+Article\s+\d+)', raw_leg_text)
parsed_leg_records = []

for segment in leg_segments:
    lines = [line.strip() for line in segment.split("\n") if line.strip()]
    if not lines:
        continue
    if lines[0].startswith("## Article"):
        article_id_raw = lines[0].replace("##", "").strip()
        article_id = article_id_raw.replace(" ", "-")
        article_title = lines[1].replace("###", "").strip() if len(lines) > 1 else "Statutory Rights"
    else:
        article_id = "CHAPTER_III_PREAMBLE"
        article_title = "General Modalities Context"

    parsed_leg_records.append({
        "chunk_id": f"GDPR-CH3-{article_id.upper()}",
        "article_id": article_id,
        "article_title": article_title,
        "text_content": segment.strip(),
        "legislation_source": "EU GDPR Official 2016/679",
        "scope_boundary": "Chapter 3 - Rights of the Data Subject"
    })

# Write to the compliance domain table
spark.createDataFrame(pd.DataFrame(parsed_leg_records)).write \
    .format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMA_PATH}.gdpr_statutory_legislation")

# ───────────────────────────────────────────────────────────────────────────────────────
# 2. PROCESS CORPORATE POLICY LAYER
# ───────────────────────────────────────────────────────────────────────────────────────
with open(policy_file, "r", encoding="utf-8") as f:
    raw_policy_text = f.read()

policy_segments = re.split(r'(?=\n##\s+Section\s+\d+)', raw_policy_text)
parsed_policy_records = []

for segment in policy_segments:
    lines = [line.strip() for line in segment.split("\n") if line.strip()]
    if not lines:
        continue
    if lines[0].startswith("## Section"):
        section_id_raw = lines[0].replace("##", "").strip()
        section_id = section_id_raw.split(":")[0].replace(" ", "_").lower()
        section_title = section_id_raw.split(":")[1].strip() if ":" in section_id_raw else section_id_raw
    else:
        section_id = "policy_preamble"
        section_title = "Global Policy Overview"

    parsed_policy_records.append({
        "chunk_id": f"RETAIL-POLICY-{section_id.upper()}",
        "section_id": section_id,
        "section_title": section_title,
        "text_content": segment.strip(),
        "corporate_owner": "Global Retail E-Commerce Corp",
        "document_type": "Internal Data Processing Matrix"
    })

# Write to the compliance domain table
spark.createDataFrame(pd.DataFrame(parsed_policy_records)).write \
    .format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMA_PATH}.retail_corporate_policy")

# ───────────────────────────────────────────────────────────────────────────────────────
# 3. ENABLE AUDIT TRAILS Across Delta Assets
# ───────────────────────────────────────────────────────────────────────────────────────
spark.sql(f"ALTER TABLE {SCHEMA_PATH}.gdpr_statutory_legislation SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
spark.sql(f"ALTER TABLE {SCHEMA_PATH}.retail_corporate_policy SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"Data Engineering Pipeline Complete. All assets locked into Unity Catalog schema: {SCHEMA_PATH}")

# View chunks for Chapter 3 GDPR 

In [0]:
%sql
SELECT chunk_id, article_id, article_title, text_content 
FROM main.default.gdpr_statutory_legislation 
ORDER BY article_id ASC;

# View chunks for Retail Privacy Policy

In [0]:
%sql
SELECT chunk_id, section_id, section_title, text_content 
FROM main.default.retail_corporate_policy;

### Vectorise Corporate Policy

In [0]:
%pip install sentence-transformers

In [0]:
# ============================================================================
# VECTORIZE RETAIL CORPORATE POLICY - Simple Version
# ============================================================================
from pyspark.sql.functions import expr
import pandas as pd
from openai import OpenAI

print("="*70)
print("VECTORIZING RETAIL CORPORATE POLICY")
print("="*70)

# Configuration
SOURCE_TABLE = "main.default.retail_corporate_policy"
TARGET_TABLE = "main.default.retail_corporate_policy_embeddings"

# Get API key and create client
OPENAI_API_KEY = dbutils.secrets.get(scope="openai", key="GDPR_agent")
client = OpenAI(api_key=OPENAI_API_KEY)
print("✓ API key loaded")

# Load all chunks
print("\nLoading chunks...")
chunks_df = spark.table(SOURCE_TABLE)
chunks_pdf = chunks_df.toPandas()
total_chunks = len(chunks_pdf)
print(f"✓ Loaded {total_chunks} chunks")

# Generate embeddings
print(f"\nGenerating embeddings for {total_chunks} texts...")
texts = chunks_pdf['text_content'].tolist()

response = client.embeddings.create(
    input=[text[:8000] for text in texts],  # Truncate to OpenAI limit
    model="text-embedding-3-small"
)

embeddings = [item.embedding for item in response.data]
chunks_pdf['embedding'] = embeddings

print(f"✓ Generated {len(embeddings)} embeddings")

# Save to Delta table
print("\nSaving embeddings...")
embeddings_df = spark.createDataFrame(chunks_pdf)

(embeddings_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE))

print(f"✓ Saved to {TARGET_TABLE}")

# Show sample
print("\n📊 Sample embeddings:")
display(spark.table(TARGET_TABLE).select(
    "chunk_id",
    "section_id",
    "section_title",
    expr("substring(text_content, 1, 100)").alias("text_preview"),
    expr("size(embedding)").alias("embedding_dimension")
).limit(5))

print("\n" + "="*70)
print("✅ RETAIL CORPORATE POLICY VECTORIZATION COMPLETE!")
print("="*70)

### Vectorise GDPR Chapter 3

In [0]:
# ============================================================================
# VECTORIZE GDPR CHAPTER 3 STATUTORY LEGISLATION - Simple Version
# ============================================================================
from pyspark.sql.functions import expr
import pandas as pd
from openai import OpenAI

print("="*70)
print("VECTORIZING GDPR CHAPTER 3 STATUTORY LEGISLATION")
print("="*70)

# Configuration
SOURCE_TABLE = "main.default.gdpr_statutory_legislation"
TARGET_TABLE = "main.default.gdpr_statutory_legislation_embeddings"

# Get API key and create client
OPENAI_API_KEY = dbutils.secrets.get(scope="openai", key="GDPR_agent")
client = OpenAI(api_key=OPENAI_API_KEY)
print("✓ API key loaded")

# Load all chunks
print("\nLoading chunks...")
chunks_df = spark.table(SOURCE_TABLE)
chunks_pdf = chunks_df.toPandas()
total_chunks = len(chunks_pdf)
print(f"✓ Loaded {total_chunks} chunks")

# Generate embeddings
print(f"\nGenerating embeddings for {total_chunks} texts...")
texts = chunks_pdf['text_content'].tolist()

response = client.embeddings.create(
    input=[text[:8000] for text in texts],  # Truncate to OpenAI limit
    model="text-embedding-3-small"
)

embeddings = [item.embedding for item in response.data]
chunks_pdf['embedding'] = embeddings

print(f"✓ Generated {len(embeddings)} embeddings")

# Save to Delta table
print("\nSaving embeddings...")
embeddings_df = spark.createDataFrame(chunks_pdf)

(embeddings_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE))

print(f"✓ Saved to {TARGET_TABLE}")

# Show sample
print("\n📊 Sample embeddings:")
display(spark.table(TARGET_TABLE).select(
    "chunk_id",
    "article_id",
    "article_title",
    expr("substring(text_content, 1, 100)").alias("text_preview"),
    expr("size(embedding)").alias("embedding_dimension")
).limit(5))

print("\n" + "="*70)
print("✅ GDPR STATUTORY LEGISLATION VECTORIZATION COMPLETE!")
print("="*70)

### Vector Search Update

In [0]:
# ============================================================================
# INSTALL REQUIRED LIBRARY (run this cell first)
# ============================================================================
%pip install databricks-vectorsearch --quiet
dbutils.library.restartPython()

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

print("="*70)
print("🔄 SYNCING VECTOR SEARCH INDICES")
print("="*70)

# List of indices to sync
indices_to_sync = [
    {
        "index_name": "main.default.gdpr_law_vector_index",
        "source": "GDPR Statutory Legislation"
    },
    {
        "index_name": "main.default.privacy_policy_vector_index",
        "source": "Retail Privacy Policy"
    }
]

# Sync each index
for config in indices_to_sync:
    print(f"\n🔄 Syncing {config['source']}...")
    try:
        index = vsc.get_index(
            endpoint_name="gdpr_rag_endpoint",
            index_name=config["index_name"]
        )
        
        # Trigger sync
        index.sync()
        print(f"   ✅ Triggered sync for {config['index_name']}")
        
    except Exception as e:
        print(f"   ❌ Error syncing {config['source']}: {e}")

print("\n" + "="*70)
print("✅ SYNC COMPLETE")
print("="*70)
print("\nNote: Sync is asynchronous. Check index status after ~2 minutes.")